# Kang County Mobility — Data Quality & Web Aggregation

## tl;dr

The executed cell below streams the supplied county origin–destination file, prints the observed coverage and quality checks, and produces the compact JSON used by the Pandemic Atlas mobility figures.

In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / 'app').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'analysis'))
from prepare_kang_mobility import DEFAULT_SOURCE, profile_and_aggregate

OUTPUT = ROOT / 'public/data/mobility.json'
profile = profile_and_aggregate(DEFAULT_SOURCE, OUTPUT)
meta = profile['meta']
quality = profile['quality']
print(f"{meta['validRowCount']:,} valid OD rows across {meta['countyCount']:,} counties "
      f"and {meta['stateCount']} state-level areas.")
print(f"Coverage: {meta['coverageStart']} to {meta['coverageEnd']}; "
      f"{meta['interstateObserved']:,} cumulative interstate observed travelers.")
print('Quality checks:', json.dumps(quality, indent=2))

4,051,110 valid OD rows across 3,135 counties and 51 state-level areas.
Coverage: 2020-03-12 to 2020-07-19; 246,717,476 cumulative interstate observed travelers.
Quality checks: {
  "malformedRows": 0,
  "negativeValueRows": 0,
  "zeroValueRows": 0,
  "duplicatePairsWithinOrigin": 0,
  "originBlockReentries": 0,
  "countyLabelConflicts": 0,
  "originCountyCount": 3135,
  "destinationCountyCount": 3135
}


## Context & Methods

The source contains one cumulative `observed_travelers` value per county origin–destination pair for March 12–July 19, 2020. Values are additive observations over the period and must not be interpreted as unique people.

### Key Assumptions

- Labels follow `State | County | FIPS`.
- County pairs are grouped by origin, enabling bounded duplicate checks.
- Inbound/outbound county metrics exclude within-county observations.
- Interstate flow-wheel values combine both directions for each state pair.

## Data

### Compact source and quality profile

In [2]:
summary = {**meta, **quality}
for key, value in summary.items():
    print(f"{key}: {value:,}" if isinstance(value, int) else f"{key}: {value}")

source: Kang county traveler totals
coverageStart: 2020-03-12
coverageEnd: 2020-07-19
grain: county origin-destination pair aggregated over the full period
unit: cumulative observed travelers; not unique individuals
rowCount: 4,051,110
validRowCount: 4,051,110
countyCount: 3,135
stateCount: 51
totalObserved: 4,654,693,048
intracountyObserved: 3,714,374,833
intrastateCrossCountyObserved: 693,600,739
interstateObserved: 246,717,476
malformedRows: 0
negativeValueRows: 0
zeroValueRows: 0
duplicatePairsWithinOrigin: 0
originBlockReentries: 0
countyLabelConflicts: 0
originCountyCount: 3,135
destinationCountyCount: 3,135


## Results

### Highest-volume interstate pairs and county hubs

In [3]:
print('Top interstate pairs:')
for row in profile['statePairs'][:10]:
    print(f"{row['source']} ↔ {row['target']}: {row['value']:,}")

print('\nTop cross-county hubs:')
for row in profile['counties'][:10]:
    print(f"{row['county']}, {row['state']}: {row['total']:,}")

Top interstate pairs:
North Carolina ↔ South Carolina: 5,973,955
Florida ↔ Georgia: 5,792,249
New Jersey ↔ New York: 4,327,592
New Jersey ↔ Pennsylvania: 4,088,168
Kansas ↔ Missouri: 4,024,621
Alabama ↔ Georgia: 3,726,535
Oklahoma ↔ Texas: 3,455,065
Illinois ↔ Indiana: 3,440,386
Louisiana ↔ Texas: 3,158,960
Illinois ↔ Missouri: 3,122,518

Top cross-county hubs:
Harris, Texas: 18,729,557
Dallas, Texas: 14,981,659
Cook, Illinois: 13,790,634
Los Angeles, California: 13,567,320
Tarrant, Texas: 12,950,255
New York City, New York: 12,524,819
Fulton, Georgia: 9,299,116
Orange, Florida: 8,893,030
Maricopa, Arizona: 8,618,976
Collin, Texas: 8,197,333


## Takeaways

- The web figures should use state-pair totals and county cross-county inbound/outbound totals, not a time-series encoding.
- Within-county observations are preserved in the profile but excluded from the county hub comparison.
- The published interface must label the date window and state that totals are cumulative observations rather than unique individuals.